TF-IDF IMPLEMENTATION OF AI PREFERENCE CLASSIFIER

In [ ]:
!pip install sentencepiece
!pip install scikit-learn
!pip install tensorflow
!pip install torch

In [5]:
import sentencepiece as spm

In [ ]:
#Training the tokenizer model
spm.SentencePieceTrainer.Train(
    '--input=dataset/train.csv --model_prefix=vocab --vocab_size=2000 --model_type=bpe'
)

In [7]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras import layers

import torch
import torch.nn as nn
import torch.optim as optim

import random

In [8]:
#Loading the trained model
sp = spm.SentencePieceProcessor()
sp.load('vocab.model')

True

In [9]:
dataset = pd.read_csv('dataset/train.csv')
dataset = dataset.dropna(subset=['prompt', 'response_a', 'response_b'])

In [10]:
dataset.head()

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1.0,0.0,0.0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0.0,1.0,0.0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0.0,0.0,1.0
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1.0,0.0,0.0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0.0,1.0,0.0


In [11]:
vocab = [str(i) for i in range(sp.get_piece_size())]
vectorizer = TfidfVectorizer(vocabulary=vocab)

In [12]:
class TFIDFdata():
    def __init__(self, data, test_size = 0.3, train = True):

        self.random_state = random.randint(1,1000)

        #Extracting the input
        self.prompt = data.iloc[:]['prompt']
        self.answer1 = data.iloc[:]['response_a']
        self.answer2 = data.iloc[:]['response_b']

        self.input = []

        for i in range(len(data)):
            self.input.append(vectorizer.fit_transform([self.prompt[i],self.answer1[i],self.answer2[i]]).toarray().flatten())

        #input and target values
        self.input = torch.tensor(np.array(self.input),dtype=torch.float32)
        self.target = torch.tensor(data.iloc[: ,-3:].astype(float).values,dtype=torch.float32)

        self.input_train, self.input_test = train_test_split(self.input,test_size=0.3, random_state=self.random_state)
        self.target_train, self.target_test = train_test_split(self.target,test_size=0.3, random_state=self.random_state)


In [13]:
data = TFIDFdata(dataset)

In [14]:
np.shape(data.input)

torch.Size([30000, 6000])

ADDING THE FEED FORWARD NETWORK

In [ ]:
class FFN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 3),
            nn.Sigmoid()           
        )
    def forward(self,x):
        return self.net(x)

In [24]:
model = FFN(input_size=6000)
optimizer = optim.Adam(model.parameters())
criterion = nn.BCELoss()

In [ ]:
for epoch in range(1000):
    optimizer.zero_grad()
    outputs = model(data.input_train)
    loss = criterion(outputs, data.target_train)
    loss.backward()
    optimizer.step()

    # print(outputs)
    print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    test_outputs = model(data.input_test)
    test_loss = criterion(test_outputs, data.target_test)
    
    test_preds = torch.argmax(test_outputs, dim=1) 
    test_targets = torch.argmax(data.target_test, dim=1)
    
    test_acc = (test_preds == test_targets).float().mean()
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

Test Loss: 2.5522, Test Accuracy: 0.3493
